<a href="https://colab.research.google.com/github/shresthshukla18/Intelligent-Video-Surveillance-System/blob/main/POC1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#cell 1

#=========================================
#DRIVE CONTECTIVITY
#=========================================

from google.colab import drive
drive.mount('/content/drive')

#=========================================
#PROJECT FOLDER
#=========================================

import os

BASE_PATH = "/content/drive/MyDrive/cv_project"

VIDEO_FOLDER = f"{BASE_PATH}/videos"
OUTPUT_FOLDER = f"{BASE_PATH}/output"

os.makedirs(VIDEO_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print("Project folders created.")

!pip install -q gdown

import gdown
import os

VIDEO_FOLDER = os.path.join(BASE_PATH, "videos")
os.makedirs(VIDEO_FOLDER, exist_ok=True)

# ---- YOUR VIDEOS ----
SAMPLE_VIDEOS = {
    "video1.mp4": "1DqvB6ylhhwLmAJrKkthzpOKG45ZwSJ7F",
    "video2.mp4": "1zAtHHq4_jhhzrlGeZJOgIIE62p550FvM",
    "video3.mp4": "1RH3meL47IOqV6yLN1RsG-i8B96iLLsGn",
    "video4.mp4": "14QWG6OJFnxS3i_4FGCwqsEcv4PBpLvrJ",
    "video5.mp4": "1pcvf8D9BX8Vu8hLsRLfGBJe4vqRdmDT9",
    "video6.mp4": "1XCWbyGm3gxMKrJNwlsilGHnEtZfTU-dj",
    "video7.mp4": "1G2-_jZXidpI5-c-abrEuqw6MvTLnPAqy",
    "video8.mp4": "1xCcZGk3D0NC4X3-DjF9IVRgnnRwMv56u",
    "video9.mp4": "1EHskew_JuJWFOtJJqlRogJ9ORWdywHTI",
    "video10.mp4": "18rIOMFuv91q_OhuMQXIFSx8EPM-R9Uzh",
    "video11.mp4": "1wH9BfN5bjmvabkFwXnXO10D3bIV6GDP1",
    "video12.mp4": "1_waULoLWFvv7zR-mmcHLwrr_dVGPYpyd",
    "video13.mp4": "13ZCReYB5ELM-dW9YRLd2hicIT3iO4ejO",
    "video14.mp4": "1nxn2apfEGp_bQH7fY21wPJfQ_RxoVB1o",
    "video15.mp4": "1YvdugJNJggW8b2CcRkFutfqrfDBgeOx4",
    "video16.mp4": "1hX-Fq-rzsqcqW1aSWE7M_sWkBRjz39He"
}

# ---- DOWNLOAD ----
for name, file_id in SAMPLE_VIDEOS.items():
    save_path = os.path.join(VIDEO_FOLDER, name)

    if not os.path.exists(save_path):
        print(f"Downloading {name}...")
        url = f"https://drive.google.com/uc?id={file_id}"
        gdown.download(url, save_path, quiet=False)
    else:
        print(f"{name} already exists")

#=========================================
#PIP INSTALLATION
#=========================================

!pip install ultralytics supervision opencv-python

#========================================
#IMPORT
#========================================

from ultralytics import YOLO
import supervision as sv
import cv2

print("Libraries loaded successfully")

Mounted at /content/drive
Project folders created.
video1.mp4 already exists
video2.mp4 already exists
video3.mp4 already exists
video4.mp4 already exists
video5.mp4 already exists
video6.mp4 already exists
video7.mp4 already exists
video8.mp4 already exists
video9.mp4 already exists
video10.mp4 already exists
video11.mp4 already exists
video12.mp4 already exists
video13.mp4 already exists
video14.mp4 already exists
video15.mp4 already exists
video16.mp4 already exists
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.4/217.4 kB 15.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/

In [ ]:
#cell 2

code = r"""
import cv2
import os
import numpy as np
import torch
import time
import csv
from ultralytics import YOLO
import supervision as sv

BASE_PATH = "/content/drive/MyDrive/cv_project"

VIDEO_FOLDER = f"{BASE_PATH}/videos"
OUTPUT_FOLDER = f"{BASE_PATH}/output"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ================= CONFIG =================
VERTICAL_LINE_PERCENT = 0.5
HORIZONTAL_LINE_PERCENT = 0.5

SLOW_THRESHOLD = 2
FAST_THRESHOLD = 8

MOVING_CLASSES = ["person","car","bus","truck","motorcycle","bicycle"]
# ==========================================


def run_pipeline(video_path):

    import time

    # ===== UNIQUE RUN FOLDER =====
    video_name = os.path.basename(video_path)
    video_name = os.path.splitext(video_name)[0]
    video_name = video_name.replace(" ", "_")

    run_id = f"{video_name}_run_{int(time.time())}"
    run_folder = os.path.join(OUTPUT_FOLDER, run_id)
    os.makedirs(run_folder, exist_ok=True)

    # ===== OUTPUT PATHS =====
    output_path = os.path.join(run_folder, "annotated.mp4")
    heatmap_video_path = os.path.join(run_folder, "heatmap.mp4")
    overlay_video_path = os.path.join(run_folder, "overlay.mp4")
    log_path = os.path.join(run_folder, "metrics.csv")

    print("\nRunning PoC-1 on:", video_path)

    # -------- GPU --------
    device = 0 if torch.cuda.is_available() else "cpu"
    print("Device:", "GPU" if device==0 else "CPU")

    model = YOLO("yolov8m.pt")
    tracker = sv.ByteTrack()

    cap = cv2.VideoCapture(video_path)

    fps_video = cap.get(cv2.CAP_PROP_FPS) or 25
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    LINE_X = int(width * VERTICAL_LINE_PERCENT)
    LINE_Y = int(height * HORIZONTAL_LINE_PERCENT)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    out = cv2.VideoWriter(output_path,fourcc,fps_video,(width,height))
    heatmap_writer = cv2.VideoWriter(heatmap_video_path,fourcc,fps_video,(width,height))
    overlay_writer = cv2.VideoWriter(overlay_video_path,fourcc,fps_video,(width,height))

    # ===== Tracking =====
    TRACK_CONFIRM_FRAMES = 5
    track_age = {}
    confirmed_tracks = set()
    track_history = {}

    total_tracks_created = 0
    total_tracks_lost = 0

    vertical_entry = 0
    vertical_exit = 0

    horizontal_entry = 0
    horizontal_exit = 0

    vertical_crossed_ids = set()
    horizontal_crossed_ids = set()

    peak_occupancy = 0
    peak_frame = 0

    direction_flow = {
        "left_to_right":0,
        "right_to_left":0,
        "top_to_bottom":0,
        "bottom_to_top":0
    }

    heatmap = np.zeros((height,width),dtype=np.float32)

    csv_file = open(log_path,"w",newline="")
    csv_writer = csv.writer(csv_file)

    csv_writer.writerow([
    "Frame","Inference_ms","Frame_ms","FPS",
    "Occupancy","V_Entry","V_Exit","H_Entry","H_Exit",
    "Slow_Count","Moderate_Count","Fast_Count",
    "Tracks_Created","Tracks_Lost"
    ])

    frame_counter = 0
    fps_history = []
    all_fps = []

    # ================= MAIN LOOP =================
    while True:

        ret,frame = cap.read()
        if not ret:
            break

        frame_counter += 1
        frame_start = time.time()

        heatmap *= 0.995

        # ----- Inference (CPU OPTIMIZED) -----
        t0 = time.time()
        results = model(frame, imgsz=640, conf=0.25, iou=0.5, device=device, verbose=False)[0]
        inference_time_ms = (time.time()-t0)*1000

        detections = sv.Detections.from_ultralytics(results)
        tracked = tracker.update_with_detections(detections)

        slow_count = moderate_count = fast_count = 0
        current_ids = set()

        for box,track_id,cls in zip(tracked.xyxy, tracked.tracker_id, tracked.class_id):

            x1,y1,x2,y2 = map(int,box)
            current_ids.add(track_id)

            if track_id not in track_age:
                track_age[track_id] = 1
            else:
                track_age[track_id] += 1

            # FIXED TRACK CREATION
            if track_age[track_id] == TRACK_CONFIRM_FRAMES:
                total_tracks_created += 1
                confirmed_tracks.add(track_id)

            if track_id not in confirmed_tracks:
                continue

            cx = int((x1+x2)/2)
            cy = int(y2)

            class_name = model.names[int(cls)]

            # Heatmap
            if class_name in MOVING_CLASSES and 0<=cx<width and 0<=cy<height:
                cv2.circle(heatmap,(cx,cy),8,1,-1)

            track_history.setdefault(track_id, []).append((cx,cy))
            if len(track_history[track_id])>15:
                track_history[track_id].pop(0)

            speed_label = ""

            if len(track_history[track_id]) >= 2:

                prev_cx,prev_cy = track_history[track_id][-2]
                vx,vy = cx-prev_cx, cy-prev_cy
                speed = np.sqrt(vx**2+vy**2)

                if speed < SLOW_THRESHOLD:
                    speed_label="Slow"; slow_count+=1
                elif speed < FAST_THRESHOLD:
                    speed_label="Moderate"; moderate_count+=1
                else:
                    speed_label="Fast"; fast_count+=1

                # Entry / Exit
                if track_id not in vertical_crossed_ids:
                    if prev_cy < LINE_Y and cy >= LINE_Y:
                        vertical_entry += 1
                        vertical_crossed_ids.add(track_id)
                    elif prev_cy >= LINE_Y and cy < LINE_Y:
                        vertical_exit += 1
                        vertical_crossed_ids.add(track_id)

                if track_id not in horizontal_crossed_ids:
                    if prev_cx < LINE_X and cx >= LINE_X:
                        horizontal_entry += 1
                        horizontal_crossed_ids.add(track_id)
                    elif prev_cx >= LINE_X and cx < LINE_X:
                        horizontal_exit += 1
                        horizontal_crossed_ids.add(track_id)

                # Direction
                if abs(vx)>abs(vy):
                    direction_flow["left_to_right" if vx>0 else "right_to_left"]+=1
                else:
                    direction_flow["top_to_bottom" if vy>0 else "bottom_to_top"]+=1

            cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),2)
            cv2.putText(frame,f"ID {track_id} {class_name} {speed_label}",
                        (x1,y1-8),cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,255,0),2)

        # ===== CLEANUP =====
        lost_ids = set(track_age.keys()) - current_ids

        for lost_id in lost_ids:
            if track_age[lost_id] >= TRACK_CONFIRM_FRAMES:
                total_tracks_lost += 1

            track_age.pop(lost_id,None)
            confirmed_tracks.discard(lost_id)
            track_history.pop(lost_id,None)

        occupancy = len(confirmed_tracks)

        if occupancy > peak_occupancy:
            peak_occupancy = occupancy
            peak_frame = frame_counter

        # FPS
        frame_time_ms = (time.time()-frame_start)*1000
        fps = 1000/frame_time_ms if frame_time_ms>0 else 0
        fps_history.append(fps)
        all_fps.append(fps)

        if len(fps_history)>30:
            fps_history.pop(0)

        avg_fps = sum(fps_history)/len(fps_history)

        # Heatmap (OPTIMIZED)
        heatmap_blur = cv2.GaussianBlur(heatmap,(25,25),0)
        heatmap_color = cv2.applyColorMap(
            cv2.normalize(heatmap_blur,None,0,255,cv2.NORM_MINMAX).astype(np.uint8),
            cv2.COLORMAP_JET
        )

        heatmap_writer.write(heatmap_color)
        overlay_writer.write(cv2.addWeighted(frame,0.7,heatmap_color,0.3,0))

        # Draw
        cv2.line(frame,(0,LINE_Y),(width,LINE_Y),(0,0,255),2)
        cv2.line(frame,(LINE_X,0),(LINE_X,height),(255,0,255),2)

        cv2.putText(frame,f"FPS:{avg_fps:.2f}",(10,40),0,0.7,(0,255,255),2)
        cv2.putText(frame,f"Occ:{occupancy}",(10,70),0,0.7,(255,255,0),2)
        cv2.putText(frame,f"Peak:{peak_occupancy}",(10,100),0,0.7,(255,100,0),2)

        out.write(frame)

        csv_writer.writerow([
            frame_counter, round(inference_time_ms,2), round(frame_time_ms,2),
            round(fps,2), occupancy,
            vertical_entry, vertical_exit,
            horizontal_entry, horizontal_exit,
            slow_count, moderate_count, fast_count,
            total_tracks_created, total_tracks_lost
        ])

    # ===== FINALIZE =====
    cap.release()
    out.release()
    heatmap_writer.release()
    overlay_writer.release()
    csv_file.close()

    final_heatmap = cv2.GaussianBlur(heatmap,(25,25),0)
    final_heatmap = cv2.applyColorMap(
        cv2.normalize(final_heatmap,None,0,255,cv2.NORM_MINMAX).astype(np.uint8),
        cv2.COLORMAP_JET
    )

    heatmap_image_path = os.path.join(run_folder, "heatmap.png")
    cv2.imwrite(heatmap_image_path, final_heatmap)

    avg_fps_total = sum(all_fps)/len(all_fps) if all_fps else 0

    return {
        "video": output_path,
        "heatmap_video": heatmap_video_path,
        "overlay": overlay_video_path,
        "csv": log_path,
        "heatmap_img": heatmap_image_path,
        "summary": {
            "tracks_created": total_tracks_created,
            "tracks_lost": total_tracks_lost,
            "peak_occupancy": peak_occupancy,
            "direction_flow": direction_flow,
            "avg_fps": round(avg_fps_total, 2)
        }
    }
"""

with open("pipeline.py", "w") as f:
    f.write(code)

print("pipeline.py created successfully")

pipeline.py created successfully


In [ ]:
#cell 3

!pip install streamlit pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 103.1 MB/s eta 0:00:00


In [ ]:
# cell 4

code = r"""
import streamlit as st
import os
import subprocess
import pandas as pd
import json
import time
import matplotlib.pyplot as plt
from pipeline import run_pipeline

# --- PAGE CONFIGURATION ---
st.set_page_config(page_title="Intelligent Video Surveillance", page_icon="👁️", layout="wide")

# --- CUSTOM CSS FOR CLEANER UI ---
st.markdown('''
<style>
    /* Hide default Streamlit menu and footer for a cleaner look */
    #MainMenu {visibility: hidden;}
    footer {visibility: hidden;}
    /* Soften the metric boxes */
    div[data-testid="metric-container"] {
        background-color: rgba(28, 131, 225, 0.1);
        border: 1px solid rgba(28, 131, 225, 0.1);
        padding: 5% 5% 5% 10%;
        border-radius: 10px;
    }
</style>
''', unsafe_allow_html=True)


BASE_PATH = "/content/drive/MyDrive/cv_project"

VIDEO_FOLDER = f"{BASE_PATH}/videos"
UPLOAD_FOLDER = f"{BASE_PATH}/uploads"
OUTPUT_FOLDER = f"{BASE_PATH}/output"

os.makedirs(VIDEO_FOLDER, exist_ok=True)
os.makedirs(UPLOAD_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

MAX_UPLOADS = 10
MAX_RUNS = 10

def cleanup_folder(folder_path, max_items):
    files = [os.path.join(folder_path, f) for f in os.listdir(folder_path)]
    if len(files) <= max_items:
        return
    files.sort(key=os.path.getctime)
    while len(files) > max_items:
        oldest = files.pop(0)
        if os.path.isfile(oldest):
            os.remove(oldest)
        else:
            import shutil
            shutil.rmtree(oldest)

def convert_to_web_video(input_path):
    output_path = input_path.replace(".mp4", "_web.mp4")
    if not os.path.exists(output_path):
        subprocess.run([
            "ffmpeg","-y","-i", input_path,
            "-vcodec","libx264","-acodec","aac",
            output_path
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return output_path

cleanup_folder(UPLOAD_FOLDER, MAX_UPLOADS)
cleanup_folder(OUTPUT_FOLDER, MAX_RUNS)

if "result" not in st.session_state:
    st.session_state.result = None

# ================= SIDEBAR (INPUTS) =================
with st.sidebar:
    st.title("⚙️ Control Panel")
    st.write("Upload a video or choose a sample to begin processing.")
    st.divider()

    st.subheader("📁 Upload Video")
    uploaded_file = st.file_uploader("Drop video here", type=["mp4","avi","mov"])

    if uploaded_file is not None:
        file_size_mb = len(uploaded_file.getvalue())/(1024*1024)
        if file_size_mb > 200:
            st.error("File too large (max 200MB)")
        else:
            ext = os.path.splitext(uploaded_file.name)[1].lower()
            if ext not in [".mp4",".avi",".mov"]:
                st.error("Invalid format")
            else:
                timestamp = int(time.time())
                unique_name = f"{timestamp}_{uploaded_file.name}"
                save_path = os.path.join(UPLOAD_FOLDER, unique_name)

                with open(save_path,"wb") as f:
                    f.write(uploaded_file.getbuffer())

                if st.button("▶ Run Upload", use_container_width=True, type="primary"):
                    with st.spinner("Processing video pipeline..."):
                        st.session_state.result = run_pipeline(save_path)

    st.divider()

    st.subheader("🎞️ Run Sample")
    videos = [
        v for v in os.listdir(VIDEO_FOLDER)
        if v.endswith((".mp4",".avi",".mov")) and not v.endswith("_web.mp4")
    ]

    if len(videos) == 0:
        st.warning("No sample videos found.")
    else:
        selected = st.selectbox("Select Sample Video", videos, label_visibility="collapsed")
        if st.button("▶ Run Sample", use_container_width=True, type="primary"):
            video_path = os.path.join(VIDEO_FOLDER, selected)
            with st.spinner("Processing sample video..."):
                st.session_state.result = run_pipeline(video_path)


# ================= MAIN DASHBOARD (OUTPUTS) =================
st.title("👁️ Intelligent Video Surveillance")
st.markdown("Automated crowd tracking, movement analysis, and anomaly detection.")

if not st.session_state.result:
    st.info("👈 Please run a video from the sidebar to view analytics.")

if st.session_state.result:
    result = st.session_state.result

    st.divider()

    # --- KPI METRICS ROW ---
    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Peak Occupancy", result['summary']['peak_occupancy'])
    col2.metric("Tracks Created", result['summary']['tracks_created'])
    col3.metric("Tracks Lost", result['summary']['tracks_lost'])
    col4.metric("Average FPS", f"{result['summary']['avg_fps']:.1f}")

    with st.expander("ℹ️ How to interpret these metrics"):
        st.write(
            "**Total Tracks Created** indicates how many unique object tracks were initialized. "
            "**Total Tracks Lost** represents tracked objects that disappeared. "
            "**Peak Occupancy** is the maximum crowd density in a single frame."
        )

    st.write("") # Spacer

    # --- TABS LAYOUT ---
    tab_viz, tab_graphs, tab_data = st.tabs(["🎥 Visualizations", "📈 Trend Analysis", "💾 Data & Export"])

    # 1. VISUALIZATIONS TAB
    with tab_viz:
        st.subheader("Pipeline Outputs")
        vid_col1, vid_col2 = st.columns(2)

        with vid_col1:
            st.markdown("**Annotated Tracking**")
            st.video(convert_to_web_video(result["video"]))
            with st.expander("About Tracking"):
                st.write("Shows real-time detection. Bounding boxes represent objects, and numbers are assigned track IDs. IDs remain consistent unless tracking is interrupted.")

            st.markdown("**Movement Heatmap**")
            st.video(convert_to_web_video(result["heatmap_video"]))
            with st.expander("About Heatmaps"):
                st.write("Visualizes movement intensity. Warmer colors (red/yellow) show high activity, cooler colors (blue) show less active regions.")

        with vid_col2:
            st.markdown("**Overlay Integration**")
            st.video(convert_to_web_video(result["overlay"]))
            with st.expander("About Overlay"):
                st.write("Combines tracking and heatmap information to allow simultaneous observation of individual movement and crowd behavior.")

            st.markdown("**Final Heatmap Summary**")
            st.image(result["heatmap_img"], use_column_width=True)

    # 2. GRAPHS TAB
    with tab_graphs:
        df = pd.read_csv(result["csv"])

        col_g1, col_g2 = st.columns(2)

        with col_g1:
            st.subheader("Crowd Density Over Time")
            fig1 = plt.figure(figsize=(6, 4))
            plt.plot(df["Frame"], df["Occupancy"], color="#1f77b4", linewidth=2)
            plt.xlabel("Frame", fontsize=9)
            plt.ylabel("Occupancy", fontsize=9)
            plt.grid(True, linestyle='--', alpha=0.6)
            st.pyplot(fig1)

            st.subheader("Performance (FPS)")
            fig2 = plt.figure(figsize=(6, 4))
            plt.plot(df["Frame"], df["FPS"], color="#ff7f0e", linewidth=2)
            plt.xlabel("Frame", fontsize=9)
            plt.ylabel("FPS", fontsize=9)
            plt.grid(True, linestyle='--', alpha=0.6)
            st.pyplot(fig2)

        with col_g2:
            st.subheader("Track Behavior Analysis")
            fig3 = plt.figure(figsize=(6, 4))
            plt.plot(df["Frame"], df["Tracks_Created"], label="Tracks Created", color="#2ca02c", linewidth=2)
            plt.plot(df["Frame"], df["Tracks_Lost"], label="Tracks Lost", color="#d62728", linewidth=2)
            plt.xlabel("Frame", fontsize=9)
            plt.ylabel("Count", fontsize=9)
            plt.legend()
            plt.grid(True, linestyle='--', alpha=0.6)
            st.pyplot(fig3)

            # Tracking Stability Box
            created = df["Tracks_Created"].iloc[-1]
            lost = df["Tracks_Lost"].iloc[-1]
            st.info(f"**Tracking Gap (Instability Indicator):** {created - lost}\n\n*A larger gap may indicate frequent occlusion or objects lingering in the scene.*")

            with st.expander("ℹ️ Read more about tracking stability"):
                st.write(
                    "Track IDs are dynamically assigned by the tracking algorithm (ByteTrack). "
                    "When an object is detected, the tracker compares it with previously seen objects based on position and motion. "
                    "If tracking fails due to occlusion, the ID is dropped (Lost Track). When found again, a new ID is generated (Created Track)."
                )

    # 3. DATA & EXPORT TAB
    with tab_data:
        st.subheader("Movement Flow Summary")
        flow = result['summary']['direction_flow']
        flow_col1, flow_col2, flow_col3, flow_col4 = st.columns(4)
        flow_col1.metric("Left → Right", flow.get('left_to_right', 0))
        flow_col2.metric("Right → Left", flow.get('right_to_left', 0))
        flow_col3.metric("Top ↓ Bottom", flow.get('top_to_bottom', 0))
        flow_col4.metric("Bottom ↑ Top", flow.get('bottom_to_top', 0))

        st.divider()

        st.subheader("Export Artifacts")
        d_col1, d_col2, d_col3 = st.columns(3)

        with open(result["video"], "rb") as f:
            d_col1.download_button("📥 Annotated Video", f.read(), os.path.basename(result["video"]), "video/mp4", use_container_width=True)

        with open(result["heatmap_video"], "rb") as f:
            d_col2.download_button("📥 Heatmap Video", f.read(), os.path.basename(result["heatmap_video"]), "video/mp4", use_container_width=True)

        with open(result["overlay"], "rb") as f:
            d_col3.download_button("📥 Overlay Video", f.read(), os.path.basename(result["overlay"]), "video/mp4", use_container_width=True)

        d_col4, d_col5, d_col6 = st.columns(3)
        with open(result["heatmap_img"], "rb") as f:
            d_col4.download_button("🖼️ Heatmap Image", f.read(), os.path.basename(result["heatmap_img"]), "image/png", use_container_width=True)

        with open(result["csv"], "rb") as f:
            d_col5.download_button("📊 Raw CSV Data", f.read(), os.path.basename(result["csv"]), "text/csv", use_container_width=True)

        summary_json = json.dumps(result["summary"], indent=4)
        d_col6.download_button("📄 Summary JSON", summary_json, "summary.json", "application/json", use_container_width=True)

        st.subheader("CSV Preview")
        st.dataframe(df.head(100), use_container_width=True)
"""

with open("dashboard.py", "w") as f:
    f.write(code)

print("dashboard.py updated safely with elegant UI layout")

dashboard.py updated safely with elegant UI layout


In [ ]:
!ls


dashboard.py  drive  pipeline.py  sample_data


In [ ]:
# cell 5

import os
import time
import subprocess
import re

print("Starting Streamlit server...")
# 1. Run Streamlit quietly in the background
subprocess.Popen([
    "streamlit", "run", "dashboard.py",
    "--server.port", "8501",
    "--server.headless", "true"
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

time.sleep(5) # Give Streamlit a few seconds to boot up

# 2. Download cloudflared if it isn't already there
if not os.path.exists("cloudflared-linux-amd64"):
    print("Downloading Cloudflared...")
    os.system("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")
    os.system("chmod +x cloudflared-linux-amd64")

print("Starting Cloudflare Tunnel...")
# 3. Run Cloudflare in the background and save its messy logs to a text file
os.system("nohup ./cloudflared-linux-amd64 tunnel --url http://localhost:8501 > tunnel_logs.txt 2>&1 &")

time.sleep(8) # Wait for the tunnel to negotiate with the server

# 4. Search the messy logs for the clean URL and print it
print("\n" + "="*50)
print("🚀 YOUR DASHBOARD IS READY!")
print("="*50)

found_url = False
with open("tunnel_logs.txt", "r") as f:
    for line in f:
        if "trycloudflare.com" in line:
            # Extract just the URL from the log line
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if match:
                print(f"\n👉 Click this link to open: {match.group(0)}\n")
                found_url = True
                break

if not found_url:
    print("\n⚠️ Still waiting for the URL... Try running this specific block again in a few seconds:")
    print("!grep -o 'https://.*\.trycloudflare\.com' tunnel_logs.txt")

Starting Streamlit server...
Starting Cloudflare Tunnel...

🚀 YOUR DASHBOARD IS READY!

👉 Click this link to open: https://bonus-circumstances-scholar-marilyn.trycloudflare.com

